In [2]:
# ============================================================
# CELL 0 — Setup & load the BRONZE (raw) tables
# ============================================================
# Bronze = raw data exactly as it landed from the Copy job (the CSV
# pulled from C:\EIA_Data and written into the Lakehouse).
# We read it here so we can clean/reshape it in the steps below.

import pandas as pd

# Read the raw bronze table straight from OneLake (Fabric's storage).
# The abfss://... path is the Lakehouse Tables location for this workspace.
energy_data = pd.read_parquet("abfss://9c1314b1-eaf8-45c0-880d-c22bba32675a@onelake.dfs.fabric.microsoft.com/2d4f360a-ca85-492c-979c-6f0491297b02/Tables/dbo/long_energy_data_bronze")

# Also read the silver table if it already exists (cleaned output from a
# previous run). Note: on the FIRST ever run this line will fail because
# the silver table hasn't been created yet (it's created down in Cell 5).
energy_data_silver = pd.read_parquet("abfss://9c1314b1-eaf8-45c0-880d-c22bba32675a@onelake.dfs.fabric.microsoft.com/2d4f360a-ca85-492c-979c-6f0491297b02/Tables/dbo/long_energy_data_silver")


StatementMeta(, 157cd94e-df35-4474-a1cb-2d0bc0f403e9, 4, Finished, Available, Finished, False)

In [3]:
# ============================================================
# CELL 1 — Quick look at the raw data
# ============================================================
# display() renders the dataframe as an interactive table so you can
# eyeball the structure before transforming it. Inspection only — no changes.
display(energy_data)


StatementMeta(, 157cd94e-df35-4474-a1cb-2d0bc0f403e9, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 32db8c3f-7368-4e9f-b81a-1c67e66f171e)

In [4]:
# ============================================================
# CELL 2 — Blank out the metric/category rows (prep for country fill-down)
# ============================================================
# The source file is hierarchical: a COUNTRY name sits on its own row,
# and the metric rows beneath it (Production, NGPL, etc.) describe that
# country. This CASE blanks out every metric row, leaving only the
# country names in description_2. That sets up the "fill country down"
# step in Cell 3.
#
# Each WHEN ... THEN "" replaces a known metric label with an empty string;
# ELSE Description keeps whatever's left (the country names).

energy_data = spark.sql("""
SELECT description, month, value,
    CASE 
        WHEN Description LIKE '%Production%' THEN ""
        WHEN Description LIKE '%Total petroleum and other liquids (Mb/d)%' THEN ""
        WHEN Description LIKE '%Crude oil, NGPL, and other liquids (Mb/d)%' THEN ""
        WHEN Description LIKE '%Crude oil including lease condensate (Mb/d)%' THEN ""
        WHEN Description LIKE '%NGPL (Mb/d)%' THEN ""
        WHEN Description LIKE '%Other liquids (Mb/d)%' THEN ""
        WHEN Description LIKE '%Refinery processing gain (Mb/d)%' THEN ""
        ELSE Description
    END AS description_2
FROM long_energy_data_bronze
""").toPandas()   # .toPandas() pulls the Spark result into a pandas dataframe

display(energy_data)


StatementMeta(, 157cd94e-df35-4474-a1cb-2d0bc0f403e9, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, daf567cc-c931-416b-869a-ced6baa70aa8)

In [5]:
# ============================================================
# CELL 3 — Attach the country to every row (the "fill down")
# ============================================================
# Build a True/False mask: True for metric rows (anything containing
# "Production" or "(Mb/d)"), False for country-name rows.
#   - r'...'  = raw string so \( escapes the parenthesis correctly
#   - na=False = treat any missing description as "not a metric"
mask_is_metric = energy_data['description'].str.contains(r'Production|\(Mb/d\)', regex=True, na=False)

# .where(~mask) keeps the description ONLY on country rows (sets metric
# rows to NaN), then .ffill() carries each country name DOWN into the
# metric rows beneath it. ffill() MUST have parentheses or it stores the
# method instead of running it.
energy_data['country'] = energy_data['description'].where(~mask_is_metric).ffill()

# Keep just the columns we need going forward.
energy_data_new = energy_data[['description', 'month', 'value', 'country']]

print(energy_data_new)


StatementMeta(, 157cd94e-df35-4474-a1cb-2d0bc0f403e9, 7, Finished, Available, Finished, False)

                                               description       month  \
0                                                    World  1973-01-01   
1                                               Production  1973-01-01   
2                 Total petroleum and other liquids (Mb/d)  1973-01-01   
3                    Crude oil, NGPL, and other liquids...  1973-01-01   
4                        Crude oil including lease cond...  1973-01-01   
...                                                    ...         ...   
1179019                                         Production  2022-02-01   
1179020           Total petroleum and other liquids (Mb/d)  2022-02-01   
1179021              Crude oil, NGPL, and other liquids...  2022-02-01   
1179022                  Crude oil including lease cond...  2022-02-01   
1179023                                        NGPL (Mb/d)  2022-02-01   

           value                       country  
0           None                         World  
1           N

In [1]:
# ============================================================
# CELL 4 — Sanity-check the cleaned data
# ============================================================
# .describe() shows summary stats (count, mean, min, max, etc.).
# Quick check that values look reasonable and the row count is what you expect.
energy_data_new.describe()


StatementMeta(, 99ba140b-361f-4253-b83f-57f38e180629, 3, Finished, Available, Finished, False)

NameError: name 'energy_data_new' is not defined

In [9]:
# ============================================================
# CELL 5 — Write the SILVER (cleaned) table
# ============================================================
# Silver = cleaned, reshaped data: description, month, value, country.
# Convert the pandas dataframe back to a Spark dataframe so we can save
# it as a managed Lakehouse table.
energy_data_new = spark.createDataFrame(energy_data_new)

# mode("Overwrite") replaces the table each run (full refresh, not append).
energy_data_new.write.mode("Overwrite").saveAsTable("long_energy_data_silver")


StatementMeta(, 157cd94e-df35-4474-a1cb-2d0bc0f403e9, 11, Finished, Available, Finished, False)

In [10]:
# ============================================================
# CELL 6 — Build the GOLD fact table
# ============================================================
# fact_world_energy_data = the main numeric table the semantic model reads.
# CAST(value AS DOUBLE) forces the value column to a number — this is the
# fix for the earlier Delta "failed to merge fields 'value'" error, which
# happened when value was sometimes text and sometimes numeric.
#
# NOTE (bug): .write....saveAsTable() returns None, so new_energy_data is
# None and the display() below will show nothing useful. If you want to see
# the result, run the SELECT on its own and display THAT, then write
# separately. The write itself works fine.
new_energy_data = spark.sql("""
SELECT description, country, month,
       CAST(value AS DOUBLE) AS value
FROM long_energy_data_silver
""").write.mode("Overwrite").saveAsTable("fact_world_energy_data")

display(new_energy_data)   # will display None — see note above


StatementMeta(, 157cd94e-df35-4474-a1cb-2d0bc0f403e9, 12, Finished, Available, Finished, False)

In [34]:
# ============================================================
# CELL 7 — Build the dim_country dimension table
# ============================================================
# A dimension table = the distinct list of countries, used to relate to
# the fact table in the semantic model (star schema).
#
# Same display(None) note as Cell 6: saveAsTable returns None, so the
# display below shows nothing. The table is still written correctly.
dim_country = spark.sql("""
SELECT DISTINCT country
FROM long_energy_data_silver
""").write.mode("Overwrite").saveAsTable("dim_country")

display(dim_country)   # will display None — table is still created


StatementMeta(, ff4ffb79-cd10-44d4-b28d-595f4f5b3418, 36, Finished, Available, Finished, False)

In [44]:
# ============================================================
# CELL 8 — Build the dim_prod_description dimension table
# ============================================================
# Distinct list of product/metric descriptions, EXCLUDING any value that
# is actually a country name (NOT IN the set of countries). This keeps the
# product dimension clean — only real metric labels, no country rows.
#
# CAUTION: NOT IN returns NO ROWS if the subquery contains any NULLs. If
# country can be null, guard it: WHERE description NOT IN
#   (SELECT country FROM long_energy_data_silver WHERE country IS NOT NULL)
#
# Same display(None) note as above: saveAsTable returns None.
dim_description = spark.sql("""
SELECT DISTINCT description AS product_description
FROM long_energy_data_silver
WHERE description NOT IN 
    (SELECT country 
    FROM long_energy_data_silver)
""").write.mode("Overwrite").saveAsTable("dim_prod_description")

display(dim_description)   # will display None — table is still created


StatementMeta(, ff4ffb79-cd10-44d4-b28d-595f4f5b3418, 46, Finished, Available, Finished, False)

# EIA Energy Data — ETL Notebook

This notebook transforms the raw EIA energy CSV (landed by the Copy job)
into a clean star schema for reporting.

**Flow:** Bronze (raw) -> Silver (cleaned) -> Gold (fact + dimensions)

| Layer | Table | Built in |
|-------|-------|----------|
| Bronze | `long_energy_data_bronze` | Copy job (upstream) |
| Silver | `long_energy_data_silver` | Cell 5 |
| Gold (fact) | `fact_world_energy_data` | Cell 6 |
| Gold (dim) | `dim_country` | Cell 7 |
| Gold (dim) | `dim_prod_description` | Cell 8 |

**Key transforms:** blank metric rows (Cell 2) -> fill country down (Cell 3)
-> cast value to DOUBLE (Cell 6).
